# Homework 9 - ST - 554 Big data analysis

## Author: Yefrid Cordoba

In this homework, we will model the compressive strength of concrete based on the amount of each component per cubic meter of material(concrete). The dataset *Concrete Compressive Strength* from the UC Irvine Machine Learning Repository will be used. We will fit three models: a multiple linear regression (MLR), a lasso regression, and a random forest — and compare their performance using the root mean squared error (RMSE) to determine the best model.

We initialize the sparl session.

In [1]:
import pandas as pd

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').appName('my_app').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 22:18:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/13 22:18:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/13 22:18:41 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/13 22:18:41 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/13 22:18:41 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/04/13 22:18:41 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
26/04/13 22:18:41 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
26/04/13 22:18:41 WARN Utils: Service 'SparkUI' coul

### Splitting the data, metrics, and models

The data is going to be imported as `pd.DataFrame`, and then turned into a sparkSQL dataframe

In [9]:
Concrete = pd.read_excel("Concrete_Data.xls", engine='xlrd')
Concrete = spark.createDataFrame(Concrete)
Concrete.columns

['Cement (component 1)(kg in a m^3 mixture)',
 'Blast Furnace Slag (component 2)(kg in a m^3 mixture)',
 'Fly Ash (component 3)(kg in a m^3 mixture)',
 'Water  (component 4)(kg in a m^3 mixture)',
 'Superplasticizer (component 5)(kg in a m^3 mixture)',
 'Coarse Aggregate  (component 6)(kg in a m^3 mixture)',
 'Fine Aggregate (component 7)(kg in a m^3 mixture)',
 'Age (day)',
 'Concrete compressive strength(MPa, megapascals) ']

Now we rename the columns

First we create a list with the new names for the variables, in this step we start preparing the column names for the use of MLlib which requires the response variable to be called `label`, while the vector with the explanatory variables or `features` is going to be created later.

In [13]:
names = ['cement', 'slag', 'ash', 'water', 'plasticizer', 'coarse', 'fine', 'age', 'label']

In [14]:
Concrete = Concrete.toDF(*names)

In [15]:
Concrete.show(5)

+------+-----+---+-----+-----------+------+-----+---+------------------+
|cement| slag|ash|water|plasticizer|coarse| fine|age|             label|
+------+-----+---+-----+-----------+------+-----+---+------------------+
| 540.0|  0.0|0.0|162.0|        2.5|1040.0|676.0| 28|       79.98611076|
| 540.0|  0.0|0.0|162.0|        2.5|1055.0|676.0| 28|61.887365759999994|
| 332.5|142.5|0.0|228.0|        0.0| 932.0|594.0|270|40.269535256000005|
| 332.5|142.5|0.0|228.0|        0.0| 932.0|594.0|365|      41.052779992|
| 198.6|132.4|0.0|192.0|        0.0| 978.4|825.5|360|      44.296075096|
+------+-----+---+-----+-----------+------+-----+---+------------------+
only showing top 5 rows
